### Plan agreed with Claude -- to review before writing any code

#### What the question actually asks for

"At what time-interval would you be least certain about how many runs there will be in
the game? (Choose a 1-minute time interval -- there can be many answers here)."

So the deliverable is **a specific one-minute window**, quoted as a timestamp, not a
method. The "many answers" hint tells us different reasonable measures will select
different minutes, and that showing that disagreement -- with the reasoning for each --
is the point rather than a weakness. We will therefore report the winning minute under
each measure separately, and then nominate one as the headline answer.

#### Universe: the KXMLBTOTAL chain, 11 strikes

`KXMLBTOTAL` is runs scored in the game by both teams combined, which is exactly what
the question asks about. `F5TOTAL` covers only five innings, `TEAMTOTAL` covers one
team, and `RFI` covers one inning, so none of them answer it. Strikes run n = 2 to 12,
giving 11 markets.

#### Step 1 -- survival curve to PMF

Each contract is a survival probability S(n) = P(runs >= n). Differencing adjacent
strikes gives the probability mass function. With strikes only spanning 2 to 12, the
result has 12 buckets, two of which are censored lumps:

| bucket | formula | size observed in this dataset |
| --- | --- | --- |
| runs <= 1 | 1 - S(2) | 3.0% - 3.5% |
| runs = n, for n = 2..11 | S(n) - S(n+1) | the interior |
| runs >= 12 | S(12) | 14.5% - 15.5% |

The head lump is small enough to ignore. The tail lump is not -- roughly 15% of the
mass sits in an open-ended bucket, and we cannot know how it splits across 12, 13, 14+.
This is the single biggest reason the choice of metric below matters.

Q1 established there are no monotonicity violations anywhere in this chain, so every
differenced bucket is guaranteed non-negative and the PMF is always well formed. We do
not need to clip or renormalise.

**The two functions this needs.** The walk calls these at every event where the gate
passes, so they are written once and kept small:

1. `get_total_ladder(cache)` -- pull the KXMLBTOTAL strikes out of the cache and return
   them as an array ordered by strike ascending, S(2) through S(12). It selects only the
   `KXMLBTOTAL` chain (not F5TOTAL, TEAMTOTAL or RFI, whose names also start with
   `KXMLB`), parses the strike number off the end of each ticker, sorts by it, and
   returns the 11 mid prices in that order. It should assert that exactly 11 strikes
   came back, because a silently short ladder would produce a PMF that looks fine and is
   wrong.

2. `ladder_to_pmf(S)` -- turn that survival ladder into the 12-bucket PMF by
   differencing adjacent strikes:

   ```
   pmf = [1 - S[0]]  +  [S[i] - S[i+1] for i in 0..9]  +  [S[10]]
       =  P(runs <= 1),  P(runs = 2..11),  P(runs >= 12)
   ```

   In numpy this is `np.concatenate([[1 - S[0]], -np.diff(S), [S[-1]]])`. The sum
   telescopes to exactly 1 by construction, so it needs no renormalising, and Q1's
   result that this chain is monotone everywhere means no bucket can come out negative.
   Assert both anyway -- they are one line each, and if either ever fails it means the
   ladder was assembled in the wrong order.

Everything downstream -- entropy, variance, E[runs] -- is computed from the output of
`ladder_to_pmf`, so the survival-to-PMF conversion lives in exactly one place.

#### Step 2 -- the three metrics, and why these

**(1) Entropy of the PMF -- how wide is the market's belief?**  `H = -sum p*log2(p)`,
in bits.

This is the information-theoretic definition of uncertainty, which is what the question
is asking about. Its decisive advantage here is that entropy is invariant to what the
outcomes are *called* -- it only cares how mass is spread across buckets. So the
`runs >= 12` lump is simply one bucket among twelve and **never has to be assigned a
numeric value**. That removes the tail problem entirely rather than papering over it.

The cost of that same property: entropy does not know that 5 runs is nearer to 6 than to
12. A distribution on {5,6} and one on {2,12} have identical entropy but very different
uncertainty about *how many* runs. That is the one thing variance knows and entropy does
not, which is why variance is kept below as a sensitivity check rather than deleted.

**(2) Volatility of E[runs] -- how hard is the market revising?**  Within each minute,
the standard deviation (and the high-low range) of E[runs] = sum[n*p(n)]

Entropy measures how wide the market's belief is; this measures how unstable it is. If
E[runs] sits at 7.3 for an hour and then thrashes between 7.2 and 7.5 inside one
minute, that minute is one where the market is actively disagreeing with itself.
Because the whole dataset is pregame -- no runs have been scored, so nothing can update
the distribution except lineups, weather and order flow -- we expect the *level*
measures to be nearly flat and the *churn* measure to carry most of the discrimination.
Diagnostics below confirm this.

E[runs] does require a numeric value for the `>= 12` bucket. We will place it at 13 and
report the sensitivity of the answer to 12 / 13 / 14, the same way Q1 reported
sensitivity to the freshness threshold.

**(3) Mean staleness of the PMF -- how well do we even observe the market?**  Walking
forward in time, keep a cache with one entry per strike holding the timestamp at which
we last saw a quote for it. At any instant, each strike's age is the current timestamp
minus its last-seen timestamp, and the PMF's staleness is the **average** of those 11
ages.

This measures something genuinely different from (1) and (2): not the market's
uncertainty about runs, but *our* uncertainty about where the market is. Both are
legitimate readings of "least certain", and the question inviting many answers is
licence to include it. A minute in which no strike has been quoted for several minutes
is a minute in which we could not confidently price anything, which is a real and
practical form of uncertainty.

*Why average and not max.* Consider two edge cases. In 1-minute interval A, we saw 10 of
the 11 strikes quoted just 1 minute ago, and the 11th strike is a laggard where we saw
its quote 10 minutes ago. Max would call this whole PMF 10 minutes stale -- but in
reality, we can be pretty sure where the 11th strike should be priced, given information
about the other 10 strikes (some interpolation based on how historically those 11
strikes relate to each other should solve for the 11th missing strike). Now consider
1-minute interval B, where all 11 strikes are missing and all of them were last seen 7
minutes ago. Max would rank this PMF as being 7 minutes stale, which is fresher and more
up to date than A. But I have seen zero strikes quoted for the past 7 minutes, and I
could argue convincingly well that I know less about where the market is in case B.
Average gets both right: A scores 1.8 minutes and B scores 7.0.

Q1 supports the interpolation argument in case A directly: the chain was internally
consistent, correctly nested, with no arbitrage anywhere, and even the never-traded PIT
strikes were perfectly coherent. That is the signature of one maker pricing the whole
ladder off a single fitted distribution, which is exactly the condition under which 10
observed strikes pin down the 11th.

A mass-weighted version -- weighting each strike by the probability mass it governs, so
a stale S(7) counts for more than a stale S(12) -- is a sensible refinement to add
later. Start with the simple average.

**Sensitivity check, not a headline metric: variance of the PMF.**
`Var = sum n^2*p(n) - (sum n*p(n))^2`. Reported alongside entropy to show the two agree,
but not used as the primary answer, because variance weights by squared distance from
the mean and roughly 15% of the mass is in an open-ended tail bucket. The answer would
partly reflect our invented tail placement rather than the market.

#### A measure considered and dropped: bid-ask spread width

We considered bid-ask spread width as a possible candidate -- makers widen their quotes
when they are uncertain -- but given that **96.4% of book rows on this chain are 1 cent
wide and the remaining 3.6% are 2 cents**, with nothing wider, there is no variation to
exploit. A measure taking two distinct values across 21,543 observations cannot
meaningfully rank 360 minutes, so it is not used.

#### Step 3 -- when is the PMF even defined?

Entropy, E[runs] and variance all require **all 11 strikes simultaneously** -- a ladder
cannot be differenced if it is only partly observed. But the strikes do not update
together: they are quoted at different times, so at most instants some are current and
others are minutes old. Evaluating on a fixed clock grid would silently forward-fill
arbitrarily old quotes into the calculation, which is the thing Q1 established we should
not do.

Instead, the PMF is **only defined when every one of the 11 strikes is fresher than a
staleness threshold**, set to **10 seconds** by default and exposed as a parameter so
the whole analysis can be re-run at other values. Note that this gate is a *maximum*
over the 11 ages, whereas metric (3) reports their *mean* -- see the note in Step 4 for
why the two aggregations are doing different jobs. Same discipline as the Q1 arbitrage
check: only compare quotes we actually observed close together in time.

Coverage at various thresholds, measured on this dataset:

| threshold | valid instants | share of chain events | minutes with >= 1 valid observation |
| --- | --- | --- | --- |
| <= 1s | 3,573 | 9.5% | 202 / 360 (56.1%) |
| <= 2s | 4,415 | 11.8% | 207 / 360 (57.5%) |
| <= 5s | 6,516 | 17.4% | 223 / 360 (61.9%) |
| **<= 10s** | **9,961** | **26.5%** | **259 / 360 (71.9%)** |
| <= 30s | 24,452 | 65.1% | 321 / 360 (89.2%) |
| <= 60s | 31,715 | 84.5% | 339 / 360 (94.2%) |

At the 10-second default, **101 of the 360 minutes have no valid PMF at all**, so
entropy and E[runs] movement are undefined there and are reported as missing rather
than filled.

Coverage is deliberately not the thing being maximised. Loosening the threshold to 30s
would raise coverage to 321 of 360 minutes, but it buys that coverage by admitting
half-minute-old quotes into a distribution calculation and then calling the result the
market's belief -- which is exactly the error the gate exists to prevent. 10 seconds is
the default for that reason, and the parameter is exposed so the sensitivity of the
answer to it can be reported rather than assumed away.

#### The shape of the answer: two tiers of uncertainty

The gate above is not just a data-hygiene step, it decides the structure of the answer.
There are two different kinds of not-knowing here, and they are not on the same scale.

**Tier 1 -- the market is unobservable.** If the last quotes we have for the 11 strikes
are far enough in the past, we cannot say where the market is pricing the expected
number of runs. Not "we can say it imprecisely" -- we literally cannot produce a number,
because there is no PMF to compute one from. These are the most uncertain intervals in
the dataset, and metric (3) is what identifies them. This is where a trader's judgement
differs from a statistician's: the temptation is to carry the stale quotes forward and
keep reporting a variance or an entropy as though nothing were wrong, which produces a
confident-looking number describing a market we cannot actually see. Refusing to compute
it is the more honest answer, and the refusal itself is the finding.

**Tier 2 -- the market is observable, and we can argue about how wide or how unstable
it is.** Among the intervals where all 11 strikes were fresh enough to derive a PMF, we
then rank on entropy, on variance, and on how much E[runs] moved. These are finer
distinctions drawn between minutes we could actually see.

So the answer is reported in that order: first the intervals where no PMF exists at all,
then the ranking among the intervals where one does.

#### A note on how the three metrics differ in kind

Entropy and variance are **snapshot** quantities. Each is computed from a single PMF at
a single instant, and mapping them to a one-minute interval just means averaging the
snapshots that fall inside it. A high-entropy minute is one where, at the moments we
could see the market, its belief was spread widely across outcomes.

E[runs] is **not** a measure of uncertainty on its own. A market pricing E[runs] = 7.4
is not more or less certain than one pricing 6.1 -- it is simply forecasting a different
number. The value carries no information about confidence. E[runs] only becomes a
measure of uncertainty when we look at how it **moves**: how much it wiggles inside a
minute, where it spikes or crashes, how far it travels relative to its usual step size.
A minute in which E[runs] is dragged from 7.30 to 7.44 and back is a minute in which the
market is actively disagreeing with itself about the forecast, and that is the sense in
which it is uncertain.

The wrong way to capture that movement is to measure it *inside* each one-minute bin.
E[runs] does not exist for a large share of the window -- whenever some strike is too
stale for a PMF to be constructed at all -- and the market does not move fast enough for
a single minute to contain meaningful variation. Measured on this dataset, the
within-minute standard deviation of E[runs] is computable in only 252 of the 360
minutes, and **it is exactly zero in 243 of those (96%)**; the within-minute high-low
range is exactly zero in 97% of minutes, and its largest value anywhere is 0.020 runs
against a full-window range of 0.135. A statistic that is zero for 97% of the bins
cannot rank them.

The right way is to treat E[runs] as a **time series over the whole six hours**. As the
walk moves from `start_time` to `end_time`, E[runs] is computed whenever the staleness
test passes and stored against its timestamp, leaving NaN wherever the PMF could not be
built. That yields one series showing how the market's forecast wandered across the
session, with visible holes where we could not see it. From that series we look at where
it actually moved: plot it, then locate the intervals containing the largest changes
from one valid observation to the next. Movement is therefore measured **between**
intervals, as a first difference along the series, not **within** an interval as a
dispersion statistic. The level of E[runs] is carried only as context, never as a
ranking measure.

#### Step 4 -- walk-forward logic over 1-minute intervals

1. Take `start_time` and `end_time` as the min and max of `recv_ts_utc` across the
   `KXMLBTOTAL` chain. Here that is 17:35:34 to 23:34:41 UTC, 5.99 hours, giving
   **360 one-minute bins**. Use `recv_ts_utc`, not exchange time, because that is when
   the information was actually available to us.
2. Build left-closed, right-open bins: `[start, start+1min)`, `[start+1min,
   start+2min)`, ... through `end_time`. Label each bin by its left edge.
3. Walk the book feed once in chronological order, holding a cache keyed by strike,
   each entry storing that strike's latest quote and the timestamp at which we saw it --
   exactly as the Q1 arbitrage engine does. Ages are always derived from those
   timestamps, never stored, so there is one source of truth to keep updated.
   Evaluate on each arriving chain event rather than on a fixed clock grid -- the feed
   is event-driven, so this samples the market when it actually moved. The dict holds
   the last quote seen per strike and its timestamp; it is a lookup of what we last
   observed, not a resampled or filled time series.
4. At each event, derive the age of all 11 strikes from the cache -- age is
   `current event timestamp - last_seen[strike]`. Record the mean age unconditionally;
   that is metric (3), and it is defined at every instant after warm-up. If and only if the
   **maximum** age is within the freshness threshold (all 11 strikes less than 10
   seconds old) do we call `get_total_ladder` and `ladder_to_pmf` from Step 1 to build
   the PMF, then compute entropy, variance and E[runs] from it.
   Append each result keyed on its timestamp, so the walk itself produces a tidy pandas
   frame with one column per metric.
   *Note on the two uses of staleness.* Both a max and a mean over the same 11 ages
   appear here, and they are doing different jobs rather than contradicting each other.
   The **max is a gate**: the PMF requires all 11 strikes at once, so its validity is
   decided by the weakest leg, and one strike older than the threshold means there is no
   PMF at all no matter how fresh the other ten are. That is deliberate -- we would
   rather report nothing than guess at E[runs] from a ladder we cannot fully see. The
   **mean is a description**: it says how old our picture of the chain is on average,
   and it is what ranks the Tier 1 minutes against each other. Max cannot do that job,
   because among minutes that have already failed the gate it is driven by whichever
   single strike lagged worst -- case A above, where ten fresh strikes and one laggard
   would be scored as badly as eleven strikes all gone quiet. So: max decides *whether*
   we can compute anything, mean measures *how well* we were seeing the market.

5. Reduce to one row per one-minute interval. The result is a DataFrame indexed by the
   360 minute bins with three columns -- `E_runs`, `entropy`, `variance` -- each taking
   the last valid observation inside that minute, plus a fourth column `mean_staleness`
   that is always populated. **Where no valid PMF could be built anywhere inside the
   minute, the first three columns are NaN.** Those NaNs are not missing data to be
   patched; they are the primary result.
6. Report in two tiers.
   - *Tier 1:* the minutes whose `E_runs`, `entropy` and `variance` are all NaN, ranked
     by `mean_staleness`. These are the intervals where the market could not be observed
     well enough to state an expected number of runs at all, and they are the most
     uncertain intervals in the dataset.
   - *Tier 2:* among the minutes that do carry values, rank by entropy (widest belief)
     and separately by the size of the change in `E_runs` from the previous valid minute
     (largest repricing). Note where the two agree and where they disagree, since the
     question invites more than one answer.

#### Assumptions, stated up front

- **Mid prices.** Price at any instant is `(best_bid + best_ask) / 2`. The chain is
  1 cent wide 96.4% of the time so the mid is a fair summary, and Q1 already established
  that mids are the right screen for chain-level structure.
- **What "using a price" means here.** There is no forward fill in the pandas sense --
  nothing is resampled onto a timestamp index and carried forward. We hold a dict with
  one entry per strike containing the last quote we actually saw for it and when we saw
  it. When all 11 strikes pass the staleness test -- meaning we last saw a quote for
  every one of them within the past 10 seconds -- we recognise those 11 probabilities as
  fresh enough to use, and compute E[runs], entropy and variance from them. If even one
  strike fails the test, no PMF is computed at that instant.

  This matters because the feed does not report every change: Q1 found that 66% of
  trades land inside book-feed gaps of more than 5 seconds. The staleness test is what
  bounds our exposure to changes we did not see, and metric (3) reports that exposure
  directly rather than hiding it.
- **Warm-up.** The first bin cannot be evaluated until all 11 strikes have been quoted
  at least once. Bins before that are dropped, not zero-filled.
- **Quiet bins.** A minute containing no quotes at all on the chain is still evaluated
  for metric (3), using the running ages -- that is precisely the case metric (3) exists
  to flag, so dropping such bins would discard the signal.
- **Tail placement.** Only metrics that need a number for the `>= 12` bucket (E[runs],
  variance) are affected. Default 13, with 12 and 14 reported as sensitivity. Entropy
  needs no such assumption.
- **Pregame only.** The window ends at 23:34 UTC and first pitch is 23:40 UTC, so no
  runs are scored anywhere in this dataset. Whatever minute we choose, it is a minute of
  pregame uncertainty, and the write-up should say so rather than imply we observed
  in-game repricing.

### Diagnostics measured before coding -- these are the validation targets

Everything below was computed directly from the parquet files while the plan above was
being settled, using the same walk the code will implement: one pass in `recv_ts_utc`
order over the `KXMLBTOTAL` chain, a cache of the last quote seen per strike, and the
PMF built only when all 11 strikes are within the freshness threshold. When the
functions in the next cell are written, **they must reproduce these numbers exactly**.
Any deviation means the implementation drifted from the plan.

#### Shape of the window

| quantity | value |
| --- | --- |
| window (`recv_ts_utc`) | 2026-08-05 17:35:34 to 23:34:41 UTC |
| span | 5.99 hours |
| first pitch | 23:40 UTC -- the entire dataset is pregame |
| `KXMLBTOTAL` strikes | 11, n = 2 through 12 |
| chain events after warm-up | 37,551 |
| one-minute bins | 360 |

#### Gate coverage, by freshness threshold

| threshold | valid instants | share of chain events | minutes with >= 1 valid observation |
| --- | --- | --- | --- |
| <= 1s | 3,573 | 9.5% | 202 / 360 (56.1%) |
| <= 2s | 4,415 | 11.8% | 207 / 360 (57.5%) |
| <= 5s | 6,516 | 17.4% | 223 / 360 (61.9%) |
| **<= 10s (default)** | **9,961** | **26.5%** | **259 / 360 (71.9%)** |
| <= 30s | 24,452 | 65.1% | 321 / 360 (89.2%) |
| <= 60s | 31,715 | 84.5% | 339 / 360 (94.2%) |

At the 10-second default: **9,961 valid instants, 259 minutes carrying a value, and 101
all-NaN Tier 1 minutes.**

#### Metric ranges at the 10s gate

| measure | min | max | total swing |
| --- | --- | --- | --- |
| entropy (bits) | 3.4048 | 3.4657 | 1.8% |
| E[runs] | 7.3000 | 7.4350 | 1.8% -- a range of 0.135 runs |
| variance | 11.4860 | 11.9100 | 3.6% |
| mean staleness, per minute | 1.58s | 81.86s | -- |

The market's belief is close to frozen across six hours, which is what a pregame window
with no observable events should look like. Consequence for the write-up: the
highest-entropy minute wins by a hair, not by a mile, so the honest claim is "belief
width was essentially constant and here is where it was widest", not "the market was
much less certain at this minute".

#### Why within-minute dispersion of E[runs] was rejected

| within-minute statistic | result |
| --- | --- |
| minutes where std is computable (>= 2 valid obs) | 252 / 360 |
| of those, std exactly 0 | **243 (96%)** |
| within-minute high-low range exactly 0 | **250 of 252 (97%)** |
| largest within-minute range anywhere | 0.020 runs |
| E[runs] range across the full window | 0.135 runs |

#### Other facts relied on in the plan

| fact | value | where it is used |
| --- | --- | --- |
| head lump P(runs <= 1) = 1 - S(2) | 3.0% - 3.5% | small enough to ignore |
| tail lump P(runs >= 12) = S(12) | 14.5% - 15.5% | why entropy beats variance |
| book rows on this chain 1 cent wide | 96.4% (rest are 2 cents) | why spread width was dropped |
| per-strike median quote gap | 0.1 - 0.3s | all strikes update together when busy |
| per-strike p90 quote gap | 9.0s (strike 8) to 53.1s (strike 2) | the quiet strikes are head and tail |
| worst gap on any single strike | 6.8 minutes | bounds how extreme staleness can get |
| trades landing in book-feed gaps > 5s | 66% (from Q1) | the feed misses changes |

### Function design -- to review before any code is written

Twelve functions in four layers. Nothing below reads a file or mutates global state; the
only entry points that touch the raw frame are `walk_total_chain` and
`threshold_sensitivity`. Two helpers already written for Q1 are reused rather than
duplicated: `market_short_name` (strips the game stamp off `native_id`) and
`parse_chain_and_strike` (returns `("TOTAL", 7)` for `KXMLBTOTAL-7`).

#### Call graph

```
threshold_sensitivity(books, thresholds)
└── for each threshold:
    ├── walk_total_chain(books, freshness_seconds, tail_value)   <-- the one pass
    │   ├── total_chain_strikes(books)          once, up front
    │   ├── pmf_bucket_values(strikes, tail)    once, up front
    │   └── per chain event:
    │       ├── get_total_ladder(cache, strikes)      -> S(2)..S(12)
    │       ├── ladder_to_pmf(S)                      -> 12 buckets
    │       ├── pmf_entropy(pmf)                      -> bits
    │       └── pmf_mean_and_variance(pmf, values)    -> (E[runs], var)
    └── minute_frame(events)                          -> 360 rows

minute_frame(events)
├── tier1_minutes(minutes)      the all-NaN minutes, ranked by staleness
├── tier2_minutes(minutes)      the minutes with values, ranked two ways
└── plot_e_runs(minutes)        the 6-hour series with its holes
```

#### Layer 1 -- chain extraction

```python
def total_chain_strikes(books):
    """Discover the KXMLBTOTAL strike ladder present in the book feed.

    Selects only the KXMLBTOTAL chain. This matters: KXMLBF5TOTAL, KXMLBTEAMTOTAL
    and KXMLBRFI all begin with 'KXMLB', and a loose substring match would silently
    pull in strikes belonging to a different random variable. Uses
    parse_chain_and_strike from utils, which names the chain explicitly, and keeps
    only chain == 'TOTAL'.

    Input : books -- DataFrame of order book rows, needs column 'native_id'
    Output: sorted list of int strikes, e.g. [2, 3, ..., 12]

    Asserts the strikes are contiguous with no gaps. A missing middle strike would
    make differencing silently wrong rather than raise.
    """

def get_total_ladder(cache, strikes):
    """Pull the survival ladder out of the cache, ordered by strike ascending.

    Input : cache   -- dict {short_name: (mid, last_seen_ts)}, the walk's state
            strikes -- the list from total_chain_strikes
    Output: np.ndarray of 11 mid prices, [S(2), S(3), ..., S(12)]

    Asserts every strike is present. A short ladder produces a PMF that looks
    perfectly well formed and is wrong, so this fails loudly rather than returning
    something shorter.
    """
```

#### Layer 2 -- distribution math (pure functions, no pandas)

```python
def ladder_to_pmf(S):
    """Difference a survival ladder into a probability mass function.

    S(n) = P(runs >= n), so adjacent differences give P(runs = exactly n), with a
    censored lump at each end that the ladder cannot resolve:

        pmf = [1 - S[0]]  +  [S[i] - S[i+1]]  +  [S[-1]]
            =  P(runs <= 1),   P(runs = 2..11),   P(runs >= 12)

    Input : S   -- np.ndarray of 11 survival probabilities, strike ascending
    Output: np.ndarray of 12 bucket probabilities

    The sum telescopes to exactly 1, so no renormalising. Q1 proved this chain is
    monotone everywhere, so no bucket can be negative. Both asserted anyway -- one
    line each, and either failing means the ladder was assembled out of order.
    """

def pmf_bucket_values(strikes, tail_value = 13.0):
    """The numeric run count each PMF bucket is taken to represent.

    Needed only by the moment calculations. Entropy never calls this, which is the
    entire reason entropy is the primary metric: the open-ended P(runs >= 12)
    bucket holds ~15% of the mass and has no defensible numeric value.

    Input : strikes    -- [2, ..., 12]
            tail_value -- what to call the '>= 12' bucket, default 13.0
    Output: np.ndarray of 12 values, [1, 2, 3, ..., 11, tail_value]

    The head lump 'runs <= 1' is assigned 1. It holds 3.0-3.5% of the mass and the
    0-versus-1 split is unknowable, but at that size it cannot move the answer.
    """

def pmf_entropy(pmf):
    """Shannon entropy of the PMF, in bits: -sum(p * log2(p)).

    Zero-probability buckets contribute nothing and are skipped rather than
    producing -inf. Invariant to what the buckets are called, so it needs no tail
    assignment at all.

    Input : pmf -- np.ndarray of 12 bucket probabilities
    Output: float, entropy in bits
    """

def pmf_mean_and_variance(pmf, values):
    """E[runs] and Var[runs] together, since both need the same bucket values.

    Input : pmf    -- np.ndarray of 12 bucket probabilities
            values -- np.ndarray of 12 numeric run counts from pmf_bucket_values
    Output: (mean, variance) as floats

    Var = sum(v^2 * p) - mean^2. Both figures inherit the tail_value assumption,
    which is why the sensitivity run varies it over 12 / 13 / 14.
    """
```

#### Layer 3 -- the walk

```python
def walk_total_chain(books, freshness_seconds = 10.0, tail_value = 13.0):
    """One chronological pass over the KXMLBTOTAL chain, event by event.

    At each arriving chain event: update that strike's cache entry with its new mid
    and timestamp, then derive every strike's age as
    (this event's timestamp - its last_seen). Mean age is recorded unconditionally.
    The PMF is built only if max age <= freshness_seconds -- max gates, mean
    describes.

    Nothing is evaluated until all 11 strikes have been seen at least once
    (warm-up). No resampling and no forward fill: the cache is a lookup of what we
    last observed, and the gate decides whether that is recent enough to use.

    Input : books             -- full order book DataFrame
            freshness_seconds -- the gate, default 10.0
            tail_value        -- passed to pmf_bucket_values, default 13.0
    Output: DataFrame indexed by recv_ts_utc, one row per chain event, columns:
                mean_staleness  float, seconds, always populated after warm-up
                max_staleness   float, seconds, always populated after warm-up
                entropy         float or NaN
                variance        float or NaN
                E_runs          float or NaN

    Expected on this dataset at the default: 37,551 rows, of which 9,961 carry a
    non-NaN PMF.
    """
```

#### Layer 4 -- aggregation and reporting

```python
def minute_frame(events):
    """Reduce the event-level frame to one row per one-minute interval.

    Entropy, variance and E_runs take the LAST valid observation inside the minute
    -- the closest analogue to a closing print, and it keeps the first-difference
    series interpretable as minute-to-minute change. A mean would blur the very
    repricing we are trying to locate. mean_staleness is the mean over all events
    in the minute and is always populated.

    Where no valid PMF exists anywhere inside the minute, the first three columns
    are NaN. Those NaNs are the primary result, not missing data to patch.

    Input : events -- the DataFrame from walk_total_chain
    Output: DataFrame indexed by minute (360 rows), columns:
                E_runs, entropy, variance, mean_staleness, n_valid
    """

def tier1_minutes(minutes):
    """The minutes where no PMF could be built at all, ranked most uncertain first.

    These are the intervals where the chain was too stale to state an expected
    number of runs -- not imprecisely, but at all -- and they are the top-level
    answer to Q2. Ranked by mean_staleness descending, because max cannot rank
    them: among minutes that already failed the gate it is driven by whichever
    single strike lagged worst, which is the case-A situation the plan rejects.

    Input : minutes -- the DataFrame from minute_frame
    Output: DataFrame of the all-NaN minutes, sorted by mean_staleness descending,
            carrying mean_staleness and n_valid for context

    Expected on this dataset at the default gate: 101 rows.
    """

def tier2_minutes(minutes):
    """Among the minutes that do carry a PMF, rank them two ways.

    (a) by entropy descending -- the widest belief over outcomes
    (b) by |change in E_runs from the previous valid minute| descending -- the
        largest repricing. This is a first difference ALONG the series, not a
        dispersion statistic within the bin, because within-minute dispersion is
        exactly zero in 97% of minutes on this data.

    Input : minutes -- the DataFrame from minute_frame
    Output: DataFrame of the valued minutes with an added d_E_runs column, plus the
            two rankings

    Expected on this dataset at the default gate: 259 rows.
    """

def plot_e_runs(minutes):
    """Plot E[runs] across the six hours, with the gaps left as gaps.

    The holes are the point -- they are the Tier 1 intervals -- so the series is
    NOT interpolated across them. Tier 1 minutes are shaded, so the reader sees
    where the market was unobservable rather than inferring it from missing ink.

    Input : minutes -- the DataFrame from minute_frame
    Output: None, draws the figure
    """

def threshold_sensitivity(books, thresholds = (1.0, 2.0, 5.0, 10.0, 30.0, 60.0)):
    """Re-run the whole walk at several freshness thresholds and summarise.

    The 10-second default is a judgement call, so the answer's dependence on it is
    reported rather than assumed away -- the same discipline as the Q1 arbitrage
    sensitivity table.

    Input : books      -- full order book DataFrame
            thresholds -- seconds to test
    Output: DataFrame, one row per threshold, columns:
                valid_instants, pct_of_events, minutes_with_value, tier1_minutes,
                entropy_min, entropy_max, E_runs_min, E_runs_max

    Must reproduce the coverage table in the diagnostics cell above.
    """
```

#### What is deliberately not a function

Tail-value sensitivity (12 / 13 / 14) needs no separate function -- `walk_total_chain`
already takes `tail_value`, so it is a loop over three calls in the notebook. Entropy is
unaffected by it, which is exactly the point worth showing.